## Influxdb3 Python basic usage

This jupyter notebook illustrates the basics of writing and querying data with an Influxdb3 database using the Influxdb3 python client.  Code cells need to be run step by step in the order in which they are presented.

1. Start by setting up the basic connection values that match your server or account.

In [ ]:
%env INFLUXDB_HOST=http://localhost:8181
%env INFLUXDB_TOKEN=<READ_WRITE_TOKEN>
%env INFLUXDB_DATABASE=my_db
from influxdb_client_3 import InfluxDBClient3, Point, WritePrecision, InfluxDBError, WriteOptions, write_client_options

2. Next, setup a basic sensor class for generating test data.  This simple class will generate readings using the built-in Influxdb3 Python `Point` class.  Using the `Point` class for writes is recommended in basic applications even though the client `write()` method handles other types of data.

In [ ]:
class Sensor:

    def __init__(self, location, model, id):
        self._location = location
        self._model = model
        self._id = id

    def point_reading(self, _measurement, temperature, humidity, pressure, timestamp) -> Point:
        return (Point(_measurement)
                .tag("location", self._location)
                .tag("model", self._model)
                .tag("id", self._id)
                .field("temperature", temperature)
                .field("humidity", humidity)
                .field("pressure", pressure)
                .time(timestamp)
                )


3. Create a client instance.  Please note that this client is instantiated using default values only.  WriteOptions and QueryOptions can also be added at this step.  For simplicity of illustration they have been omitted.

In [ ]:
import logging
import os

logging.basicConfig(level=logging.INFO)

logging.info("Using\n"
            "INFLUXDB_HOST={}\n"
            "INFLUXDB_DATABASE={}".format(os.environ["INFLUXDB_HOST"], os.environ["INFLUXDB_DATABASE"]))

client = InfluxDBClient3(
    host= os.environ["INFLUXDB_HOST"],
    token=os.environ["INFLUXDB_TOKEN"],
    database=os.environ["INFLUXDB_DATABASE"]
)

logging.info("Have client {}".format(client))


4. Generate data points and write them to the database.

In [ ]:
import random
from datetime import datetime, timezone, timedelta

sensors = [Sensor("entry_hall", "R2D2", "a2026001"),
           Sensor("conf01", "C3PO", "a2026002"),
           Sensor("conf02", "R2D2", "a2026004"),
           Sensor("hall_west", "C3PO", "a2026005"),
           Sensor("hall_east", "ROBBIE", "b2025255"),
           ]

measurement = "sensor_test"
samplesize = 1000
interval = 10 # seconds

now = datetime.now(timezone.utc)
ts = now - timedelta(seconds=(interval * samplesize))

data = []

while ts < now:
    for sensor in sensors:
        data.append(
            sensor.point_reading(
                _measurement=measurement,
                temperature=random.uniform(10.0,35.0),
                humidity=random.uniform(50.0,100.0),
                pressure=random.uniform(26.0,32.0),
                timestamp=ts
            )
        )
    ts = ts + timedelta(seconds=interval)
try:
   client.write(data)
   logging.info(f"Write successful!")
except InfluxDBError as e:
    logging.error("InfluxDB error: {}".format(e))


Now query

In [ ]:
sql = f"SELECT time, location, model, temperature FROM {measurement} WHERE location = 'hall_east' ORDER BY time DESC"

table = []

try:
    table = client.query(query=sql, language="sql", mode="pandas")
    print(table)
except InfluxDBError as e:
    print("InfluxDB error: {}".format(e))

Sample results plot below.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

temps = table.get("temperature")
times = table.get("time")

plt.plot(times, temps)
plt.title(f"Temperature in hall_east")
plt.xlabel("Time")
plt.ylabel("Temperature")
plt.show()